# Random forest with // UCI-Bank-Marketing

In [2]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/Rawanmalharbi/UCI-Bank-Marketing/main/bank-full.csv"
df = pd.read_csv(URL, sep=";")

print(df.head())
print(df.shape) # (45211, 17)
print(df.describe())
print(df.info())

# print(df.isnull().sum())
# print(df.duplicated().sum()) # no missing or duplicates

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  
(45211, 17)
                age        balance           day      duration      campaign  \
count  45211.000000

## baseline model

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns="y")
y = df["y"]

X = pd.get_dummies(X, drop_first=True) # there are some categorical features should convert to numerical

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# train and fit
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

# predict score
print(rf.score(X_test, y_test)) # 0.9068893066460245
print(rf.score(X_train, y_train)) # 0.9999723512497235

0.9068893066460245
0.9999723512497235


## prepruned model // used grid search to find best params

In [8]:
from sklearn.model_selection import GridSearchCV

grid = {
    'max_depth': [3, 5, 7, 10, 15, 20],
    'n_estimators': [5, 10, 50, 100],
    'max_features': [0.5, 'sqrt', 'log2', None],
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=grid,
    cv=5,
    n_jobs=-1,
)

gs.fit(X_train, y_train)

print(f"Best params: {gs.best_params_}") # Best params: {'max_depth': 15, 'max_features': 0.5, 'n_estimators': 100}
print(f"Best score: {gs.best_score_}") # Best score: 0.9063814855582543

Best params: {'max_depth': 15, 'max_features': 0.5, 'n_estimators': 100}
Best score: 0.9063814855582543


### selecting important features

In [9]:
imp = pd.Series(gs.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False)

print(imp.head(10))

duration            0.329150
poutcome_success    0.098207
age                 0.085535
balance             0.079359
day                 0.066429
pdays               0.052689
campaign            0.026610
housing_yes         0.024886
previous            0.019027
month_mar           0.017913
dtype: float64


In [12]:
features_3 = ["duration", "poutcome_success", "age"]
features_4 = ["duration", "poutcome_success", "age", "balance"]

for name, features in [("Top 3", features_3), ("Top 4", features_4)]:
    model = RandomForestClassifier(
        max_depth=15,
        max_features=0.5,
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train[features], y_train)

    print(name)
    print("Train:", model.score(X_train[features], y_train))
    print("Test:", model.score(X_test[features], y_test))

# Top 3
# Train: 0.9399192656491927
# Test: 0.8914077186774301
# Top 4
# Train: 0.9459743419597434
# Test: 0.8967156916952339

Top 3
Train: 0.9399192656491927
Test: 0.8914077186774301
Top 4
Train: 0.9459743419597434
Test: 0.8967156916952339


## compare Decision tree and Random forest

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

dt = DecisionTreeClassifier(random_state=42)
rf = gs.best_estimator_

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Decision Tree": dt,
    "Random Forest": rf
}

for name, model in models.items():
    model.fit(X_train, y_train)

    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    cv_score = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1
    ).mean()

    print(name)
    print(f"Train Accuracy: {train_score}")
    print(f"Test Accuracy: {test_score}")
    print(f"CV Accuracy: {cv_score}")

# Decision Tree
# Train Accuracy: 1.0
# Test Accuracy: 0.8786906999889417
# CV Accuracy: 0.8747511339164058
# Random Forest
# Train Accuracy: 0.9688951559389516
# Test Accuracy: 0.9054517306203693
# CV Accuracy: 0.9055244560945267

Decision Tree
Train Accuracy: 1.0
Test Accuracy: 0.8786906999889417
CV Accuracy: 0.8747511339164058
Random Forest
Train Accuracy: 0.9688951559389516
Test Accuracy: 0.9054517306203693
CV Accuracy: 0.9055244560945267


In [14]:
# important features
imp_dt = pd.Series(dt.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print(imp_dt.head(3))
# duration    0.279326
# balance     0.102603
# age         0.095748

imp_rf = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp_rf.head(3))
# duration            0.329516
# poutcome_success    0.105831
# age                 0.084500
# both of them has same imp features

duration    0.279326
balance     0.102603
age         0.095748
dtype: float64
duration            0.329516
poutcome_success    0.105831
age                 0.084500
dtype: float64
